# Verify Outer Join Datasets

Checks that:
- Patients with omics data have real values
- Patients without omics have zeros + missingness indicator = 1
- OS/OS.time are correct for both groups

In [2]:
import pandas as pd
from pathlib import Path

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

OUTPUT_DIR = SCRIPT_DIR / "MERGE_outer"
df = pd.read_csv(OUTPUT_DIR / "01_ultra_conservative.csv", index_col=0)

print(f"Shape: {df.shape}")
print(f"Total patients: {len(df)}")
print()

# Split into "had omics" vs "missing omics"
had_omics     = df[df["RNA_missing"] == 0]
missing_omics = df[df["RNA_missing"] == 1]

print(f"Patients WITH all 7 modalities:    {len(had_omics)}")
print(f"Patients WITHOUT some modalities:  {len(missing_omics)}")
print()

# Check RNA features for missing group — should be all zeros
rna_cols = [c for c in df.columns if c.startswith("RNA_") and not c.endswith("_missing")][:5]
print(f"Sample RNA columns: {rna_cols}")
print()
print("RNA values for patients WITH data (first 3):")
print(had_omics[rna_cols].head(3).to_string())
print()
print("RNA values for patients WITHOUT data (first 3):")
print(missing_omics[rna_cols].head(3).to_string())
print()

# Check missingness indicators
miss_cols = [c for c in df.columns if c.endswith("_missing")]
print(f"Missingness indicator columns: {miss_cols}")
print()
print("Indicator values for patients WITH data (first 3):")
print(had_omics[miss_cols].head(3).to_string())
print()
print("Indicator values for patients WITHOUT data (first 3):")
print(missing_omics[miss_cols].head(3).to_string())
print()

# Check OS/OS.time
print(f"Events (OS=1) in full dataset: {int(df['OS'].sum())}")
print(f"Events in had_omics:    {int(had_omics['OS'].sum())}")
print(f"Events in missing_omics: {int(missing_omics['OS'].sum())}")
print()
print(f"OS.time range: {df['OS.time'].min():.0f} – {df['OS.time'].max():.0f} days")

Shape: (1076, 452)
Total patients: 1076

Patients WITH all 7 modalities:    526
Patients WITHOUT some modalities:  550

Sample RNA columns: ['RNA_ENSG00000266925.1', 'RNA_ENSG00000228098.1', 'RNA_ENSG00000287660.1', 'RNA_ENSG00000248120.1', 'RNA_ENSG00000222308.1']

RNA values for patients WITH data (first 3):
              RNA_ENSG00000266925.1  RNA_ENSG00000228098.1  RNA_ENSG00000287660.1  RNA_ENSG00000248120.1  RNA_ENSG00000222308.1
TCGA-3C-AALI              -0.166667              -0.258471              -0.161339               -0.20795              -0.185318
TCGA-3C-AALK              -0.166667              -0.258471              -0.161339               -0.20795              -0.185318
TCGA-4H-AAAK              -0.166667              -0.258471              -0.161339               -0.20795              -0.185318

RNA values for patients WITHOUT data (first 3):
              RNA_ENSG00000266925.1  RNA_ENSG00000228098.1  RNA_ENSG00000287660.1  RNA_ENSG00000248120.1  RNA_ENSG00000222308.1

In [4]:
df = pd.read_csv("MERGE_outer/01_ultra_conservative.csv", index_col=0)
rna_cols = [c for c in df.columns if c.startswith("RNA_") and not c.endswith("_missing")]
had = df[df["RNA_missing"] == 0][rna_cols]
print("Unique rows in RNA data:", had.drop_duplicates().shape[0], "of", len(had))

Unique rows in RNA data: 168 of 526


In [6]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction")

# Check the original inner-join file
inner = pd.read_csv(BASE / "MERGE" / "01_ultra_conservative.csv", index_col=0)
rna_cols = [c for c in inner.columns if c.startswith("RNA_") and not c.endswith("_missing")]

print("Inner join shape:", inner.shape)
print("Unique RNA rows:", inner[rna_cols].drop_duplicates().shape[0], "of", len(inner))
print()

# Find duplicate RNA patterns
rna_data = inner[rna_cols]
duplicated_mask = rna_data.duplicated(keep=False)
print("Patients with duplicate RNA:", duplicated_mask.sum())

# Show which patients share the same RNA
dupes = inner[duplicated_mask][rna_cols[:3]]
print("\nSample duplicate group:")
print(dupes.head(6))

Inner join shape: (526, 446)
Unique RNA rows: 168 of 526

Patients with duplicate RNA: 388

Sample duplicate group:
              RNA_ENSG00000266925.1  RNA_ENSG00000228098.1  \
TCGA-3C-AALI              -0.166667              -0.258471   
TCGA-3C-AALK              -0.166667              -0.258471   
TCGA-5L-AAT1              -0.166667              -0.258471   
TCGA-5T-A9QA              -0.166667              -0.258471   
TCGA-A1-A0SF              -0.166667              -0.258471   
TCGA-A1-A0SH              -0.166667              -0.258471   

              RNA_ENSG00000287660.1  
TCGA-3C-AALI              -0.161339  
TCGA-3C-AALK              -0.161339  
TCGA-5L-AAT1              -0.161339  
TCGA-5T-A9QA              -0.161339  
TCGA-A1-A0SF              -0.161339  
TCGA-A1-A0SH              -0.161339  


In [8]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction")

# Check the RNA statistical_filtered file directly
rna_file = BASE / "RNA" / "statistical_filtered" / "rna_1_ultra_conservative_723genes.csv"
rna = pd.read_csv(rna_file, index_col=0)

print("RNA file shape:", rna.shape)
print("Sample barcodes:", rna.index[:5].tolist())
print()

# Normalize to patient IDs
def normalize_id(idx):
    parts = str(idx).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(idx)

rna_normalized = rna.copy()
rna_normalized.index = [normalize_id(i) for i in rna_normalized.index]

print("After normalization:")
print("Total rows:", len(rna_normalized))
print("Unique patient IDs:", rna_normalized.index.nunique())
print()

# Check Primary Tumor filter
tumor_mask = rna.index.str.contains(r"-01[A-Z]?$", regex=True)
print("Primary Tumor samples:", tumor_mask.sum(), "of", len(rna))
print("Sample Primary Tumor IDs:", rna[tumor_mask].index[:5].tolist())

RNA file shape: (1073, 723)
Sample barcodes: ['TCGA-3C-AAAU-01A', 'TCGA-3C-AALI-01A', 'TCGA-3C-AALJ-01A', 'TCGA-3C-AALK-01A', 'TCGA-4H-AAAK-01A']

After normalization:
Total rows: 1073
Unique patient IDs: 1073

Primary Tumor samples: 1071 of 1073
Sample Primary Tumor IDs: ['TCGA-3C-AAAU-01A', 'TCGA-3C-AALI-01A', 'TCGA-3C-AALJ-01A', 'TCGA-3C-AALK-01A', 'TCGA-4H-AAAK-01A']


In [10]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction")

# Load the MB gene list for this config
MB_DIR = BASE / "RNA" / "mb_results"
gene_file = MB_DIR / "rna_1_ultra_conservative_723genes" / "IAMB_alpha0.05_genes.txt"
genes = [l.strip() for l in gene_file.read_text().splitlines() if l.strip()]
print(f"MB selected genes: {len(genes)}")
print(f"First 5: {genes[:5]}")
print()

# Check variance of these genes in original RNA file
rna_file = BASE / "RNA" / "statistical_filtered" / "rna_1_ultra_conservative_723genes.csv"
rna = pd.read_csv(rna_file, index_col=0)

# Filter to selected genes that exist
available = [g for g in genes if g in rna.columns]
print(f"Available in RNA file: {len(available)} of {len(genes)}")
print()

rna_sel = rna[available]
print("Variance of selected genes (top 5 and bottom 5):")
var = rna_sel.var().sort_values()
print("Lowest variance:")
print(var.head(5))
print("\nHighest variance:")
print(var.tail(5))
print()
print(f"Genes with variance < 0.001: {(var < 0.001).sum()}")
print(f"Unique rows in selected genes: {rna_sel.drop_duplicates().shape[0]} of {len(rna_sel)}")

MB selected genes: 50
First 5: ['ENSG00000228098.1', 'ENSG00000287660.1', 'ENSG00000248120.1', 'ENSG00000270900.1', 'ENSG00000224357.1']

Available in RNA file: 50 of 50

Variance of selected genes (top 5 and bottom 5):
Lowest variance:
ENSG00000248764.1    1.000933
ENSG00000264767.2    1.000933
ENSG00000222744.1    1.000933
ENSG00000261511.1    1.000933
ENSG00000287660.1    1.000933
dtype: float64

Highest variance:
ENSG00000283207.1    1.000933
ENSG00000200361.1    1.000933
ENSG00000226873.1    1.000933
ENSG00000225896.1    1.000933
ENSG00000276442.1    1.000933
dtype: float64

Genes with variance < 0.001: 0
Unique rows in selected genes: 317 of 1073


In [12]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction")
rna_file = BASE / "RNA" / "statistical_filtered" / "rna_1_ultra_conservative_723genes.csv"
rna = pd.read_csv(rna_file, index_col=0)

# Check first gene - what are its actual values?
first_gene = rna.columns[0]
print(f"Gene: {first_gene}")
print(f"Min: {rna[first_gene].min():.6f}")
print(f"Max: {rna[first_gene].max():.6f}")
print(f"Mean: {rna[first_gene].mean():.6f}")
print(f"Unique values: {rna[first_gene].nunique()}")
print()

# How many genes have < 10 unique values?
low_unique = (rna.apply(lambda x: x.nunique()) < 10).sum()
print(f"Genes with <10 unique values: {low_unique} of {rna.shape[1]}")

# Overall data range
print(f"\nOverall data range: {rna.values.min():.4f} to {rna.values.max():.4f}")
print(f"Are values z-scored? (mean≈0, std≈1): mean={rna.values.mean():.4f}, std={rna.values.std():.4f}")

Gene: ENSG00000270900.1
Min: -0.273853
Max: 7.580374
Mean: 0.000000
Unique values: 12

Genes with <10 unique values: 669 of 723

Overall data range: -0.4540 to 32.7414
Are values z-scored? (mean≈0, std≈1): mean=0.0000, std=1.0000


In [14]:
from pathlib import Path
BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\RNA")

print("Folders in RNA:")
for p in sorted(BASE.iterdir()):
    if p.is_dir():
        print(f"  {p.name}/")
        for f in sorted(p.glob("*.csv"))[:3]:
            print(f"    {f.name}")

Folders in RNA:
  .ipynb_checkpoints/
  mb_results/
    summary_all_results.csv
  statistical_filtered/
    datasets_summary.csv
    outcome.csv
    rna_1_ultra_conservative_723genes.csv


In [16]:
from pathlib import Path
BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3")

print("Files in data/:")
for f in sorted((BASE / "data").glob("*RNA*")):
    print(f"  {f.name}")
for f in sorted((BASE / "data").glob("*rna*")):
    print(f"  {f.name}")
for f in sorted((BASE / "data").glob("*.tsv")):
    print(f"  {f.name}")

Files in data/:
  TCGA-BRCA.mirna.tsv
  TCGA-BRCA.mirna.tsv
  clinical.tsv
  TCGA-BRCA.clinical.tsv
  TCGA-BRCA.gene-level_ascat2.tsv
  TCGA-BRCA.methylation450.tsv
  TCGA-BRCA.mirna.tsv
  TCGA-BRCA.protein.tsv
  TCGA-BRCA.somaticmutation_wxs.tsv
  TCGA-BRCA.star_counts.tsv
  TCGA-BRCA.survival.tsv


In [18]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\olegk\Desktop\Thesis_v3")

rna_raw = pd.read_csv(BASE / "data" / "TCGA-BRCA.star_counts.tsv", 
                      sep="\t", index_col=0, nrows=5)
print("Shape (5 rows):", rna_raw.shape)
print("Index (genes or samples?):", rna_raw.index[:3].tolist())
print("Columns (first 3):", rna_raw.columns[:3].tolist())
print()
print("Value range:")
print(rna_raw.iloc[:, :5].to_string())

Shape (5 rows): (5, 1226)
Index (genes or samples?): ['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.13']
Columns (first 3): ['TCGA-D8-A146-01A', 'TCGA-AQ-A0Y5-01A', 'TCGA-C8-A274-01A']

Value range:
                    TCGA-D8-A146-01A  TCGA-AQ-A0Y5-01A  TCGA-C8-A274-01A  TCGA-BH-A0BD-01A  TCGA-B6-A1KC-01B
Ensembl_ID                                                                                                  
ENSG00000000003.15         11.737670          9.781360         13.122504         11.016808         11.000000
ENSG00000000005.6           7.721099          3.321928          0.000000          6.686501          3.807355
ENSG00000000419.13         11.042343         11.357552         11.506308         10.801708         11.074141
ENSG00000000457.14         11.036860         10.754888         12.218260         11.190442         10.857981
ENSG00000000460.17          9.131857          8.721099         10.973697         10.761551          9.550747
